In [0]:
%restart_python

In [0]:
dbutils.widgets.text("path_design_campaign", "")
path_design_campaign = dbutils.widgets.get("path_design_campaign")
print("Path Design Campaign: ", path_design_campaign)
#path_design_campaign = "/Volumes/sandbox/karthik/motif_scaffolding/experiments/rfdiffusion3/designs/rfd3_test_ajzmv"

In [0]:
import os
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib.pyplot as plt

path_boltz_folder = os.path.join(path_design_campaign, "af2_passed_boltz_input")

list_df_boltz_holo = []
for filename_boltz_holo in os.listdir(path_boltz_folder):
    if (("boltz_holo_scored" in filename_boltz_holo) and (filename_boltz_holo.endswith(".csv"))):
        filepath_boltz_holo = os.path.join(path_boltz_folder, filename_boltz_holo)
        df_boltz_holo = pd.read_csv(filepath_boltz_holo, index_col= 0)
        list_df_boltz_holo.append(df_boltz_holo) 
df_boltz_designs = pd.concat(list_df_boltz_holo, axis = 0)
df_boltz_designs.rename(columns = {'ptm' : 'ptm_boltz'}, inplace= True)
df_boltz_designs.head()

In [0]:
df_boltz_most_confident = df_boltz_designs[df_boltz_designs['model_id'] == 0]

figsize = (8, 6)
col_af2_boltz_holo_rmsd = 'rmsd_af2_boltz_holo_binder'
fig, ax = plt.subplots(figsize = figsize)
sns.histplot(data = df_boltz_most_confident, x = col_af2_boltz_holo_rmsd, bins = 100, ax = ax)
ax.set_xlabel(col_af2_boltz_holo_rmsd)
ax.set_ylabel("Number of designs")
ax.set_title(f"Distribution of {col_af2_boltz_holo_rmsd} for {df_boltz_most_confident.shape[0]} designs");

In [0]:
agg_funcs = ['median', 'mean', 'std']
df_boltz_groupedby = df_boltz_designs.groupby('design_name').agg({'ptm_boltz' : agg_funcs, 'ligand_iptm' : agg_funcs, 'protein_iptm' : agg_funcs, 'complex_plddt' : agg_funcs, 'complex_iplddt' : agg_funcs, 'complex_pde' : agg_funcs, col_af2_boltz_holo_rmsd: agg_funcs})
df_boltz_groupedby.sort_values(by = (col_af2_boltz_holo_rmsd, 'median'), ascending = True, inplace = True)
df_boltz_groupedby.reset_index(inplace = True)
df_boltz_groupedby

In [0]:
rmsd_threshold = 3
df_top_designs = df_boltz_groupedby[df_boltz_groupedby[col_af2_boltz_holo_rmsd]['median'] < rmsd_threshold]
df_top_designs.columns = ['_'.join(col).strip('_') for col in df_top_designs.columns]
print(f"Number of designs with RMSD < {rmsd_threshold}: {df_top_designs.shape[0]}")
df_top_designs

In [0]:
df_boltz_top_predictions = pd.merge(df_boltz_designs, df_top_designs, on = 'design_name', how = 'inner')
df_boltz_top_predictions.sort_values(by = [f'{col_af2_boltz_holo_rmsd}_median', 'model_id'], ascending = [True, True], inplace = True)
df_boltz_top_predictions.reset_index(drop = True, inplace = True)
df_boltz_top_predictions

In [0]:
import py2Dmol
import sys
import random
sys.path.append('/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics')
from StrucTools import *
df_boltz_top_predictions_model0 = df_boltz_top_predictions[df_boltz_top_predictions['model_id'] == 0]
random_design = random.randint(0, df_boltz_top_predictions_model0.shape[0] - 1) 
print(f"Index of random design: {random_design}")
print(f"Design name: {df_boltz_top_predictions_model0['design_name'].iloc[random_design]}")
print(f"RMSD: {df_boltz_top_predictions_model0[col_af2_boltz_holo_rmsd].iloc[random_design]}")
print(f"Confidence Score: {df_boltz_top_predictions_model0['confidence_score'].iloc[random_design]}")
path_structure = df_boltz_top_predictions_model0['path_boltz_holo_structure'].iloc[random_design]
visualize_structure(path_structure)

### Merging Back to AF2 Outputs

In [0]:
filename_af2_metrics = "af2_passed_designs.csv"
filepath_af2_metrics = os.path.join(path_boltz_folder, filename_af2_metrics)
df_af2 = pd.read_csv(filepath_af2_metrics, index_col = 0)
df_af2

In [0]:
df_boltz_holo_af2 = pd.merge(left = df_boltz_top_predictions, right = df_af2, on = 'design_name', how = 'inner')
df_boltz_holo_af2

In [0]:
df_boltz_holo_af2_designs = df_boltz_holo_af2[df_boltz_holo_af2['model_id'] == 0].reset_index(drop = True)
df_boltz_holo_af2_designs

In [0]:
path_final_metrics = os.path.join(path_design_campaign, "final_metrics")
if not os.path.exists(path_final_metrics):
    print(f"Creating directory: {path_final_metrics}")
    os.makedirs(path_final_metrics)
else:
    print(f"Directory already exists: {path_final_metrics}")

# Save the DataFrame as a CSV file
filename_passed_boltz_holo = f"af2_boltz_holo_passed_median_rmsd_less_{rmsd_threshold}.csv"
path_af2_boltz_holo_metrics = os.path.join(path_final_metrics, filename_passed_boltz_holo)
df_boltz_holo_af2_designs.to_csv(os.path.join(path_final_metrics, path_af2_boltz_holo_metrics))

In [0]:
list_paths_boltz_holo_outputs = []

# Save the full csv to the boltz folder regardless of chunking or not:
path_af2_boltz_holo_passed_in_boltz_folder = os.path.join(path_boltz_folder, filename_passed_boltz_holo)
df_boltz_holo_af2_designs.to_csv(path_af2_boltz_holo_passed_in_boltz_folder)

# If/Else condition only to determine whether to split the dataframe into chunks or not and if so, what input should be passed to next step
# If the number of passed designs is greater than 20, split the dataframe into chunks for running Boltz in parallel
if len(df_boltz_holo_af2_designs) > 20:
    for index, df_chunk in enumerate(np.array_split(df_boltz_holo_af2_designs, 6)):
        path_boltz_passed_chunk = os.path.join(path_boltz_folder, f"af2_boltz_holo_passed_designs_chunk_{index}.csv")
        df_chunk.to_csv(path_boltz_passed_chunk)
        list_paths_boltz_holo_outputs.append(path_boltz_passed_chunk)
# If the number of passed designs is less than 20, save the dataframe to a single CSV file
else:
    list_paths_boltz_holo_outputs.append(path_af2_boltz_holo_passed_in_boltz_folder)

print(f"List of paths to CSV files: {list_paths_boltz_holo_outputs}")

# Save list of paths to passed CSV files to dbutils.taskValue
dbutils.jobs.taskValues.set(key = "list_paths_design_csv", value = list_paths_boltz_holo_outputs)

# Decide whether to run Boltz2 Holo or Not
if len(df_boltz_holo_af2_designs) == 0:
    dbutils.jobs.taskValues.set(key = "run_boltz", value = "")
    print("No designs passed the filter. Skipping Boltz2 Holo.")
else:
    dbutils.jobs.taskValues.set(key = "run_boltz", value = "apo")
    print("Designs passed the filter. Running Boltz2 Apo.")